# Lantern — Cascade Hybrid Recommender: Training
**MINE4201-01 · Taller 2 · 2026-1**

Pipeline: SVD++ (CF) + post-filtro contextual + prior de popularidad + content-based cold start  
Dataset: Yelp Open Dataset (Philadelphia)

---

## Bloque A — Setup y carga de datos

In [ ]:
# Cell 1 — Imports y constantes
import json, ast, re, os, sys, warnings, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

try:
    from surprise import SVDpp, Dataset as SurpriseDataset, Reader
    SURPRISE_OK = True
    print('surprise OK')
except ImportError:
    SURPRISE_OK = False
    print('WARNING: surprise no instalado — pip install scikit-surprise')

import joblib
warnings.filterwarnings('ignore')

# ── Paths
ROOT       = Path('..')
DATA_DIR   = ROOT / 'data'
REAL_DIR   = DATA_DIR / 'real' / 'structured'   # yelp JSONs live here
ARTIFACTS  = DATA_DIR
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# ── Filtrado
MIN_USER_REVIEWS = 5
MIN_BIZ_REVIEWS  = 10

# ── SVD++
N_FACTORS    = 50
N_EPOCHS     = 20
LR_ALL       = 0.005
REG_ALL      = 0.02
RANDOM_STATE = 42

# ── Pesos híbrido
W_CF  = 0.60
W_CTX = 0.25
W_POP = 0.15

# ── Top-N
TOP_N = 50

# ── Proyección SVG (Philly bbox → 680×700 px)
SVG_W, SVG_H, SVG_PAD = 680, 700, 10
PHILLY_LAT_MIN, PHILLY_LAT_MAX = 39.87, 40.14
PHILLY_LON_MIN, PHILLY_LON_MAX = -75.28, -74.96

print(f'REAL_DIR existe: {REAL_DIR.exists()}')
print(f'ARTIFACTS: {ARTIFACTS}')

In [ ]:
# Cell 2 — Carga de businesses de Philadelphia

def _parse_attr_value(val):
    if val is None:        return None
    if val == 'True':      return True
    if val == 'False':     return False
    if val == 'None':      return None
    if isinstance(val, str):
        m = re.match(r"^u'(.+)'$", val)
        if m: return m.group(1)
        if val.startswith('{'):
            try: return ast.literal_eval(val)
            except Exception: return None
    return val

def _extract_attrs(raw):
    if not raw: return {}
    out = {}
    for k, v in raw.items():
        p = _parse_attr_value(v)
        if isinstance(p, dict):
            for sk, sv in p.items(): out[f'{k}_{sk}'] = sv
        else:
            out[k] = p
    return out

def _to_svg(lat, lon):
    if lat is None or lon is None: return None, None
    x = (lon - PHILLY_LON_MIN) / (PHILLY_LON_MAX - PHILLY_LON_MIN) * SVG_W + SVG_PAD
    y = (PHILLY_LAT_MAX - lat)  / (PHILLY_LAT_MAX - PHILLY_LAT_MIN) * SVG_H + SVG_PAD
    return round(float(x), 1), round(float(y), 1)


print('Cargando businesses Philadelphia...')
t0 = time.time()
biz_rows = []

with open(REAL_DIR / 'yelp_academic_dataset_business.json', encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        city = r.get('city', '')
        if 'hiladelphia' not in city:
            continue
        attrs    = _extract_attrs(r.get('attributes') or {})
        cat_list = [c.strip() for c in (r.get('categories') or '').split(',') if c.strip()]
        svg_x, svg_y = _to_svg(r.get('latitude'), r.get('longitude'))
        biz_rows.append({
            'business_id':    r['business_id'],
            'name':           r['name'].strip(),
            'address':        r.get('address', ''),
            'postal_code':    str(r.get('postal_code', '')).strip(),
            'latitude':       r.get('latitude'),
            'longitude':      r.get('longitude'),
            'svg_x':          svg_x,
            'svg_y':          svg_y,
            'stars':          r.get('stars'),
            'review_count':   r.get('review_count', 0),
            'is_open':        int(r.get('is_open', 0)),
            'categories':     r.get('categories', ''),
            'category_list':  cat_list,
            'category_primary': cat_list[0] if cat_list else 'Other',
            'hours':          r.get('hours') or {},
            'price_range':    attrs.get('RestaurantsPriceRange2'),
            'outdoor_seating': attrs.get('OutdoorSeating'),
            'wifi':           attrs.get('WiFi'),
            'alcohol':        attrs.get('Alcohol'),
        })

businesses_df = pd.DataFrame(biz_rows)
print(f'Philadelphia: {len(businesses_df):,}  ({time.time()-t0:.1f}s)')

In [ ]:
# Cell 3 — ZIP → Neighborhood
ZIP_TO_NEIGHBORHOOD = {
    '19102':'Center City',     '19103':'Rittenhouse',
    '19104':'University City', '19106':'Old City',
    '19107':'Washington Square','19108':'Center City',
    '19109':'Center City',     '19110':'Center City',
    '19111':'Fox Chase',       '19112':'South Philly',
    '19113':'South Philly',    '19114':'Torresdale',
    '19115':'Northeast Philly','19116':'Northeast Philly',
    '19118':'Chestnut Hill',   '19119':'Mount Airy',
    '19120':'Olney',           '19121':'North Philly',
    '19122':'North Philly',    '19123':'Northern Liberties',
    '19124':'Frankford',       '19125':'Fishtown',
    '19126':'Oak Lane',        '19127':'Manayunk',
    '19128':'Roxborough',      '19129':'East Falls',
    '19130':'Fairmount',       '19131':'West Philly',
    '19132':'North Philly',    '19133':'Kensington',
    '19134':'Kensington',      '19135':'Mayfair',
    '19136':'Holmesburg',      '19137':'Bridesburg',
    '19138':'Germantown',      '19139':'West Philly',
    '19140':'Logan',           '19141':'Germantown',
    '19142':'Southwest Philly','19143':'Southwest Philly',
    '19144':'Germantown',      '19145':'South Philly',
    '19146':'Point Breeze',    '19147':'Bella Vista',
    '19148':'South Philly',    '19149':'Mayfair',
    '19150':'Mount Airy',      '19151':'Overbrook',
    '19152':'Northeast Philly','19153':'Southwest Philly',
    '19154':'Northeast Philly',
}

businesses_df['neighborhood'] = (
    businesses_df['postal_code'].map(ZIP_TO_NEIGHBORHOOD).fillna('Philadelphia')
)

fallback_pct = (businesses_df['neighborhood'] == 'Philadelphia').mean() * 100
print(f'Fallback rate: {fallback_pct:.1f}%')
if fallback_pct > 40:
    raise RuntimeError(f'STOP: {fallback_pct:.1f}% sin neighborhood — revisar tabla ZIP')

print(businesses_df['neighborhood'].value_counts().head(15).to_string())

In [ ]:
# Cell 4 — Cargar reviews (sin texto)
print('Cargando reviews Philadelphia...')
t0 = time.time()
philly_ids = set(businesses_df['business_id'])
rev_rows = []

with open(REAL_DIR / 'yelp_academic_dataset_review.json', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % 1_000_000 == 0 and i > 0:
            print(f'  {i//1_000_000}M leidas...')
        r = json.loads(line)
        if r['business_id'] not in philly_ids:
            continue
        rev_rows.append({
            'user_id':     r['user_id'],
            'business_id': r['business_id'],
            'stars':       float(r['stars']),
            'date':        r['date'][:10],
        })

reviews_df = pd.DataFrame(rev_rows)
reviews_df['date'] = pd.to_datetime(reviews_df['date'])
print(f'Reviews Philly: {len(reviews_df):,}  ({time.time()-t0:.1f}s)')
print(f'Users:          {reviews_df.user_id.nunique():,}')

## Bloque B — Filtrado y preparación CF

In [ ]:
# Cell 5 — Umbrales N=5 / M=10
user_counts = reviews_df['user_id'].value_counts()
biz_counts  = reviews_df['business_id'].value_counts()

warm_user_ids = set(user_counts[user_counts >= MIN_USER_REVIEWS].index)
warm_biz_ids  = set(biz_counts [biz_counts  >= MIN_BIZ_REVIEWS ].index)
cold_user_ids = set(user_counts.index) - warm_user_ids
cold_biz_ids  = set(biz_counts.index)  - warm_biz_ids

reviews_warm = reviews_df[
    reviews_df['user_id'].isin(warm_user_ids) &
    reviews_df['business_id'].isin(warm_biz_ids)
].copy()

print(f'Users warm:       {len(warm_user_ids):,}  /  cold: {len(cold_user_ids):,}')
print(f'Businesses warm:  {len(warm_biz_ids):,}  /  cold: {len(cold_biz_ids):,}')
print(f'Reviews warm:     {len(reviews_warm):,}')

In [ ]:
# Cell 6 — Split temporal 80/10/10
reviews_sorted = reviews_warm.sort_values('date').reset_index(drop=True)
n = len(reviews_sorted)
train_df = reviews_sorted.iloc[:int(n * 0.80)].copy()
val_df   = reviews_sorted.iloc[int(n * 0.80):int(n * 0.90)].copy()
test_df  = reviews_sorted.iloc[int(n * 0.90):].copy()

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

In [ ]:
# Cell 7 — Checkpoint magnitudes
if SURPRISE_OK:
    reader      = Reader(rating_scale=(1.0, 5.0))
    surprise_ds = SurpriseDataset.load_from_df(
        train_df[['user_id', 'business_id', 'stars']], reader
    )
    trainset = surprise_ds.build_full_trainset()
    density  = trainset.n_ratings / (trainset.n_users * trainset.n_items) * 100

    print('=' * 55)
    print('  CHECKPOINT — MAGNITUDES')
    print('=' * 55)
    print(f'  biz Philly:       {len(businesses_df):>8,}')
    print(f'  biz warm:         {len(warm_biz_ids):>8,}  cold: {len(cold_biz_ids):,}')
    print(f'  users warm:       {len(warm_user_ids):>8,}  cold: {len(cold_user_ids):,}')
    print(f'  reviews total:    {len(reviews_df):>8,}')
    print(f'  reviews warm:     {len(reviews_warm):>8,}')
    print(f'  train/val/test:   {len(train_df):,} / {len(val_df):,} / {len(test_df):,}')
    print(f'  Trainset users:   {trainset.n_users:>8,}')
    print(f'  Trainset items:   {trainset.n_items:>8,}')
    print(f'  Trainset ratings: {trainset.n_ratings:>8,}')
    print(f'  Density:          {density:>10.4f}%')
    print(f'  Neighborhood fallback: {(businesses_df.neighborhood=="Philadelphia").mean()*100:.1f}%')
    print('=' * 55)

## Bloque C — Entrenar SVD++

In [ ]:
# Cell 8 — Entrenar SVD++
if not SURPRISE_OK:
    raise RuntimeError('Instalar scikit-surprise: pip install scikit-surprise')

print(f'Entrenando SVD++ (factors={N_FACTORS}, epochs={N_EPOCHS})...')
t0 = time.time()

model = SVDpp(
    n_factors    = N_FACTORS,
    n_epochs     = N_EPOCHS,
    lr_all       = LR_ALL,
    reg_all      = REG_ALL,
    random_state = RANDOM_STATE,
    verbose      = True,
)
model.fit(trainset)

print(f'\nEntrenado en {time.time()-t0:.0f}s')
joblib.dump(model, ARTIFACTS / 'svdpp_model.joblib')
print('Guardado: svdpp_model.joblib')

## Bloque D — Pre-cómputo top-N y explanations (usuarios warm)

In [ ]:
# Cell 9 — Popularidad + contexto

pop_df = businesses_df[['business_id', 'review_count']].copy()
pop_df['pop_score'] = MinMaxScaler().fit_transform(
    np.log1p(pop_df[['review_count']])
)
biz_pop = pop_df.set_index('business_id')['pop_score'].to_dict()

CAT_HOUR_BOOST = {
    'Breakfast & Brunch': {'morning': 1.3, 'lunch': 1.1},
    'Coffee & Tea':       {'morning': 1.4, 'afternoon': 1.2},
    'Bars':               {'dinner': 1.2, 'late_night': 1.5},
    'Italian':            {'dinner': 1.2},
    'Sushi Bars':         {'lunch': 1.1, 'dinner': 1.3},
    'Pizza':              {'lunch': 1.1, 'dinner': 1.2, 'late_night': 1.3},
}

def _ctx_score(cat_list: list, hour: int = 20) -> float:
    if   6  <= hour < 11: bucket = 'morning'
    elif 11 <= hour < 15: bucket = 'lunch'
    elif 15 <= hour < 18: bucket = 'afternoon'
    elif 18 <= hour < 23: bucket = 'dinner'
    else:                 bucket = 'late_night'
    boost = 1.0
    for c in cat_list:
        if c in CAT_HOUR_BOOST:
            boost = max(boost, CAT_HOUR_BOOST[c].get(bucket, 1.0))
    return min((boost - 1.0) / 0.5, 1.0)

biz_cats = businesses_df.set_index('business_id')['category_list'].to_dict()
print('Popularidad y contexto listos.')

In [ ]:
# Cell 10 — Pre-cómputo top-N (hora 20:00 por defecto)
print(f'Generando top-{TOP_N} para {len(warm_user_ids):,} usuarios...')
t0 = time.time()

all_biz    = list(businesses_df['business_id'])
top_n_rows = []
expl_rows  = []

user_list = list(warm_user_ids)
for idx, uid in enumerate(user_list):
    if idx % 500 == 0:
        elapsed = time.time() - t0
        rate    = idx / elapsed if elapsed > 0 else 0
        eta     = (len(user_list) - idx) / rate if rate > 0 else 0
        print(f'  {idx:,}/{len(user_list):,}  ETA {eta/60:.1f}min', end='\r')

    seen = set(train_df[train_df['user_id'] == uid]['business_id'])
    scores = []
    for bid in all_biz:
        if bid in seen:
            continue
        cf_raw = model.predict(uid, bid).est
        cf  = (cf_raw - 1.0) / 4.0
        pop = biz_pop.get(bid, 0.0)
        ctx = _ctx_score(biz_cats.get(bid, []), hour=20)
        hybrid = W_CF * cf + W_CTX * ctx + W_POP * pop
        scores.append((bid, hybrid, cf, ctx, pop))

    scores.sort(key=lambda x: -x[1])
    for bid, score, cf, ctx, pop in scores[:TOP_N]:
        top_n_rows.append({'user_id': uid, 'business_id': bid, 'score': round(score, 4)})
        expl_rows.append({
            'user_id':     uid,
            'business_id': bid,
            'cf':          round(cf  * 100),
            'ctx':         round(ctx * 100),
            'pop':         round(pop * 100),
        })

print(f'\nDone en {(time.time()-t0)/60:.1f}min  |  rows: {len(top_n_rows):,}')

## Bloque E — Modelo de contenido (Cold Start)

Para usuarios sin historial: TF-IDF sobre categorías + features numéricas (precio, popularidad, rating).  
La similitud coseno entre el perfil de gustos del usuario y el vector de cada negocio genera el ranking.

In [ ]:
# Cell 11 — Construir matriz de features de negocios
print('Construyendo feature matrix para content-based...')

cb_df = businesses_df[[
    'business_id', 'categories', 'stars', 'review_count',
    'is_open', 'price_range',
]].copy()

# TF-IDF sobre la cadena de categorías
tfidf = TfidfVectorizer(
    analyzer   = 'word',
    token_pattern = r'[A-Za-z][A-Za-z ]+',
    min_df     = 3,
    max_features = 300,
    sublinear_tf = True,
)
cat_texts = cb_df['categories'].fillna('').tolist()
tfidf_mat = tfidf.fit_transform(cat_texts)   # (n_biz, n_terms)

# Features numéricas normalizadas
scaler_cb = MinMaxScaler()
num_feats = cb_df[['stars', 'review_count', 'is_open']].copy()
num_feats['review_count'] = np.log1p(num_feats['review_count'])
num_mat = scaler_cb.fit_transform(num_feats)   # (n_biz, 3)

# Price one-hot (1..4)
def _price_int(v):
    try: return int(float(str(v)))
    except: return 2
price_vals = cb_df['price_range'].apply(_price_int).clip(1, 4)
price_oh = np.zeros((len(cb_df), 4))
for i, p in enumerate(price_vals):
    price_oh[i, p - 1] = 1.0

# Concatenar todo: TF-IDF (sparse) + num (dense) + price_oh (dense)
num_sparse   = sp.csr_matrix(num_mat)
price_sparse = sp.csr_matrix(price_oh)
feature_mat  = sp.hstack([tfidf_mat, num_sparse, price_sparse])  # (n_biz, n_features)

biz_ids_cb = cb_df['business_id'].tolist()   # índice posicional → business_id
biz_idx_cb = {bid: i for i, bid in enumerate(biz_ids_cb)}

print(f'Feature matrix: {feature_mat.shape}  (negocios × features)')
print(f'  TF-IDF terms:  {tfidf_mat.shape[1]}')
print(f'  Numéricas:     {num_mat.shape[1]}  (stars, log_reviews, is_open)')
print(f'  Price one-hot: 4')

In [ ]:
# Cell 12 — Perfiles de gusto → vector de consulta
#
# Cada perfil es un dict {category_keyword: weight} que se proyecta
# sobre el vocabulario TF-IDF + se combina con preferencias numéricas.

TASTE_PROFILES = {
    # Nombre usado en el frontend como user_id cuando no hay historial
    'new_visitor': {
        'categories': 'Restaurants Food',
        'stars_pref':  0.8,   # prefiere alta calificación (0-1)
        'price_pref':  2,     # precio medio (1-4)
        'pop_pref':    0.5,   # popularidad media
    },
    'italian_lover': {
        'categories': 'Italian Pizza Mediterranean European',
        'stars_pref':  0.85,
        'price_pref':  3,
        'pop_pref':    0.6,
    },
    'coffee_seeker': {
        'categories': 'Coffee Tea Cafe Bakeries Breakfast Brunch',
        'stars_pref':  0.75,
        'price_pref':  1,
        'pop_pref':    0.4,
    },
    'nightlife': {
        'categories': 'Bars Nightlife Cocktail Lounges Wine Beer',
        'stars_pref':  0.7,
        'price_pref':  2,
        'pop_pref':    0.7,
    },
}

def _build_query_vector(profile: dict) -> sp.csr_matrix:
    """Convierte un perfil de gusto en un vector de la misma dim que feature_mat."""
    # TF-IDF sobre las categorías preferidas
    tfidf_q = tfidf.transform([profile['categories']])   # (1, n_terms)

    # Numéricas: stars_pref, log(pop_pref * max_reviews) normalizado, is_open=1
    stars_norm = profile.get('stars_pref', 0.8)
    pop_norm   = profile.get('pop_pref', 0.5)
    num_q      = np.array([[stars_norm, pop_norm, 1.0]])     # (1, 3)
    num_q_sp   = sp.csr_matrix(num_q)

    # Price one-hot
    price_val = max(0, min(3, int(profile.get('price_pref', 2)) - 1))
    price_q   = np.zeros((1, 4))
    price_q[0, price_val] = 1.0
    price_q_sp = sp.csr_matrix(price_q)

    return sp.hstack([tfidf_q, num_q_sp, price_q_sp])


# Verificar dimensiones
test_q = _build_query_vector(TASTE_PROFILES['new_visitor'])
assert test_q.shape[1] == feature_mat.shape[1], \
    f'Dim mismatch: query {test_q.shape[1]} vs matrix {feature_mat.shape[1]}'
print(f'Query vector dim: {test_q.shape[1]}  ✓')

In [ ]:
# Cell 13 — Generar top-N cold-start para cada perfil de gusto
print('Generando recomendaciones cold-start...')

cold_rows = []

for profile_name, profile in TASTE_PROFILES.items():
    query_vec = _build_query_vector(profile)
    sims      = cosine_similarity(query_vec, feature_mat).flatten()   # (n_biz,)

    # Combinar similitud coseno con popularidad (cold start no tiene CF)
    pop_scores = np.array([biz_pop.get(bid, 0.0) for bid in biz_ids_cb])
    final_scores = 0.75 * sims + 0.25 * pop_scores

    top_indices = np.argsort(final_scores)[::-1][:TOP_N]
    for rank, idx in enumerate(top_indices):
        bid   = biz_ids_cb[idx]
        score = float(final_scores[idx])
        sim   = float(sims[idx])
        pop   = float(pop_scores[idx])
        cold_rows.append({
            'user_id':     profile_name,
            'business_id': bid,
            'score':       round(score, 4),
            'cf':          0,                        # no hay CF para cold start
            'ctx':         round(sim * 100),         # similitud de contenido → ctx
            'pop':         round(pop * 100),
        })

    top3 = [biz_ids_cb[i] for i in top_indices[:3]]
    names = businesses_df.set_index('business_id')['name']
    print(f'  {profile_name}: top-3 → {[names.get(b, b) for b in top3]}')

cold_df = pd.DataFrame(cold_rows)
print(f'\nCold-start rows: {len(cold_df):,}')

## Guardar todos los artefactos

In [ ]:
# Cell 14 — Guardar todos los artefactos
top_n_df = pd.DataFrame(top_n_rows)
expl_df  = pd.DataFrame(expl_rows)

# Unir warm + cold-start en top_n y explanations
top_n_all = pd.concat([top_n_df, cold_df[['user_id', 'business_id', 'score']]], ignore_index=True)
expl_all  = pd.concat([
    expl_df,
    cold_df[['user_id', 'business_id', 'cf', 'ctx', 'pop']],
], ignore_index=True)

top_n_all.to_parquet(ARTIFACTS / 'top_n.parquet', index=False)
expl_all.to_parquet(ARTIFACTS / 'explanations.parquet', index=False)

# business_meta para el backend
meta_cols = ['business_id', 'name', 'neighborhood', 'svg_x', 'svg_y',
             'latitude', 'longitude', 'stars', 'review_count', 'price_range',
             'categories', 'is_open', 'address']
businesses_df[meta_cols].to_parquet(ARTIFACTS / 'business_meta.parquet', index=False)

# Modelo SVD++ ya guardado en Cell 8

# Modelo content-based
joblib.dump(
    {'tfidf': tfidf, 'feature_mat': feature_mat,
     'biz_ids': biz_ids_cb, 'scaler': scaler_cb},
    ARTIFACTS / 'content_model.joblib'
)

# Perfiles de gusto pre-definidos
import json as _json
(ARTIFACTS / 'taste_profiles.json').write_text(
    _json.dumps(TASTE_PROFILES, indent=2)
)

# Pesos del híbrido
ctx_weights = {'cf': W_CF, 'ctx': W_CTX, 'pop': W_POP}
(ARTIFACTS / 'ctx_weights.json').write_text(_json.dumps(ctx_weights, indent=2))

print('Artefactos guardados:')
for p in ['top_n.parquet', 'explanations.parquet', 'business_meta.parquet',
          'svdpp_model.joblib', 'content_model.joblib',
          'taste_profiles.json', 'ctx_weights.json']:
    f = ARTIFACTS / p
    size = f.stat().st_size / 1024**2 if f.exists() else 0
    print(f'  {p:<32} {size:.1f} MB')